# LeetCode 2168 — Unique Substrings With Equal Digit Frequency bằng Python

Link đề: https://leetcode.com/problems/unique-substrings-with-equal-digit-frequency/ · Độ khó: **Medium** (🔒 Premium) · Chủ đề: Hash Table, String, Rolling Hash, Counting, Hash Function

> **Lưu ý Premium:** đây là bài Premium; đề bài trong notebook được viết theo bản mô tả công khai của doocs/leetcode. Nếu bạn có tài khoản Premium, hãy **đối chiếu lại với đề gốc** (tên hàm, ràng buộc, ví dụ).

Skeleton LeetCode:

```python
class Solution:
    def equalDigitFrequency(self, s: str) -> int:
        ...
```

Đếm số **xâu con khác nhau** mà mọi chữ số xuất hiện trong nó đều có **cùng số lần**. Hai khó khăn tách biệt: (1) kiểm tra điều kiện "tần suất bằng nhau" nhanh, (2) loại **trùng lặp** giữa các xâu con. Notebook đi từ **`Counter` cho từng xâu con `O(n³)`** → **mảng tiền tố 10 chữ số** → **đếm tăng dần với `max` và số chữ số khác nhau `O(n²)`** → **băm lăn (rolling hash)** để khử trùng không cần cắt chuỗi → **trie các hậu tố** (mỗi nút mới = một xâu con mới).

## 1. Mục tiêu học

1. Biến điều kiện "mọi chữ số xuất hiện cùng số lần" thành phép kiểm tra `O(1)`: `maxCount × distinct == length`.
2. Duyệt mọi xâu con theo kiểu "cố định đầu `i`, kéo dài đuôi `j`" và **cập nhật tăng dần** bộ đếm.
3. Khử trùng bằng `set` các xâu con, bằng **băm lăn** (cặp `(hash, length)`), hoặc bằng **trie** (mỗi xâu con khác nhau ứng với đúng một nút).
4. Hiểu rủi ro **va chạm băm** và cách giảm (modulo lớn, cơ số ngẫu nhiên, kèm độ dài).
5. Kiểm thử đối chiếu với vét cạn `Counter` trên xâu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho xâu chữ số `s`. Trả về số **xâu con khác nhau** của `s` mà trong đó **mọi chữ số có mặt** đều xuất hiện **cùng số lần**.

**Ví dụ 1:** `s = "1212"` → `5`: các xâu con thỏa là `"1", "2", "12", "21", "1212"` (`"12"` xuất hiện hai lần nhưng chỉ đếm một lần).

**Ví dụ 2:** `s = "12321"` → `9`: `"1", "2", "3", "12", "23", "32", "21", "123", "321"`.

**Ràng buộc:** `1 ≤ len(s) ≤ 1000`, `s` chỉ gồm chữ số.

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [("1212", 5), ("12321", 9)]

for s, exp in examples:
    good = sorted({s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)
                   if len(set(Counter(s[i:j]).values())) == 1}, key=lambda x: (len(x), x))
    print(f"s={s!r}: các xâu con thỏa = {good} -> {len(good)} (kỳ vọng {exp})")

s='1212': các xâu con thỏa = ['1', '2', '12', '21', '1212'] -> 5 (kỳ vọng 5)
s='12321': các xâu con thỏa = ['1', '2', '3', '12', '21', '23', '32', '123', '321'] -> 9 (kỳ vọng 9)


## 3. Kiến thức nền

### 3.1 `Counter` và tập giá trị tần suất

In [2]:
for t in ["1212", "122", "123", "1"]:
    c = Counter(t)
    print(f"{t!r}: Counter = {dict(c)}, tập tần suất = {set(c.values())}, thỏa? {len(set(c.values())) == 1}")

'1212': Counter = {'1': 2, '2': 2}, tập tần suất = {2}, thỏa? True
'122': Counter = {'1': 1, '2': 2}, tập tần suất = {1, 2}, thỏa? False
'123': Counter = {'1': 1, '2': 1, '3': 1}, tập tần suất = {1}, thỏa? True
'1': Counter = {'1': 1}, tập tần suất = {1}, thỏa? True


### 3.2 Điều kiện "bằng nhau" qua `max × distinct`

Với các số dương `c₁, …, c_k` có tổng `L` và lớn nhất `M`: `L ≤ k · M`, dấu bằng **khi và chỉ khi** mọi `cᵢ = M`. ⇒ kiểm tra `M · k == L` là đủ, và cả ba đại lượng cập nhật được trong `O(1)` khi thêm một ký tự.

In [3]:
for cnts in [[2, 2], [2, 1], [3, 3, 3], [1]]:
    L, M, k = sum(cnts), max(cnts), len(cnts)
    print(cnts, f"-> L={L}, M={M}, k={k}, M*k==L? {M * k == L}")

[2, 2] -> L=4, M=2, k=2, M*k==L? True
[2, 1] -> L=3, M=2, k=2, M*k==L? False
[3, 3, 3] -> L=9, M=3, k=3, M*k==L? True
[1] -> L=1, M=1, k=1, M*k==L? True


### 3.3 Mảng tiền tố cho từng chữ số

`pre[d][i]` = số lần chữ số `d` xuất hiện trong `s[:i]`. Số lần `d` trong `s[i:j]` là `pre[d][j] − pre[d][i]`.

In [4]:
s = "12321"
pre = [[0] * (len(s) + 1) for _ in range(10)]
for i, ch in enumerate(s):
    for d in range(10):
        pre[d][i + 1] = pre[d][i] + (ord(ch) - 48 == d)
print("pre[1] =", pre[1], "| pre[2] =", pre[2], "| số '2' trong s[1:4] =", pre[2][4] - pre[2][1])

pre[1] = [0, 1, 1, 1, 1, 2] | pre[2] = [0, 0, 1, 1, 2, 2] | số '2' trong s[1:4] = 2


### 3.4 Băm lăn (polynomial rolling hash)

`h(t) = (t₀·Bᴸ⁻¹ + t₁·Bᴸ⁻² + … + t_{L−1}) mod P`. Kéo dài thêm ký tự `c`: `h' = (h·B + c) mod P` — `O(1)`. Hai xâu khác nhau có thể trùng băm (**va chạm**); dùng `P = 2⁶¹ − 1` (số nguyên tố Mersenne), `B` ngẫu nhiên và kèm **độ dài** để xác suất va chạm cực nhỏ.

In [5]:
P = (1 << 61) - 1
B = 911382323
def h(t):
    x = 0
    for ch in t:
        x = (x * B + ord(ch)) % P
    return x
print("h('12') =", h("12"), "| h('21') =", h("21"), "| h('12') lần nữa =", h("12"))

h('12') = 44657733877 | h('21') = 45569116199 | h('12') lần nữa = 44657733877


### 3.5 Trie: mỗi xâu con khác nhau là một nút

Chèn mọi **hậu tố** `s[i:]` vào trie. Mỗi đường từ gốc tới một nút là một **tiền tố của một hậu tố** = một **xâu con**; hai xâu con giống nhau đi tới **cùng** một nút. ⇒ số nút (trừ gốc) = số xâu con khác nhau; đánh dấu nút "đã đếm" để khử trùng.

## 4. Ý tưởng nền tảng / trực giác

Tính tay `s = "1212"` (10 xâu con theo vị trí):

| xâu con | tần suất | thỏa? |
|---|---|---|
| `1`, `2`, `1`, `2` | `{1:1}` / `{2:1}` | ✓ (nhưng chỉ 2 xâu khác nhau) |
| `12`, `21`, `12` | `{1:1,2:1}` | ✓ (2 xâu khác nhau) |
| `121`, `212` | `{1:2,2:1}` / `{2:2,1:1}` | ✗ |
| `1212` | `{1:2,2:2}` | ✓ |

Khác nhau và thỏa: `1, 2, 12, 21, 1212` ⇒ 5 ✓.

Cấu trúc duyệt tự nhiên: cố định đầu `i`, kéo dài `j = i, i+1, …`. Khi thêm `s[j]`, chỉ một chữ số đổi tần suất ⇒ cập nhật `M`, `k`, `L` trong `O(1)` ⇒ kiểm tra điều kiện `O(1)`. Tổng `O(n²)` lần kiểm tra. Phần còn lại là **khử trùng**.

## 5. Cách 1 — Brute Force / baseline: `Counter` cho từng xâu con

In [6]:
def edf_brute(s):
    seen = set()
    n = len(s)
    for i in range(n):
        for j in range(i + 1, n + 1):
            t = s[i:j]
            if len(set(Counter(t).values())) == 1:
                seen.add(t)
    return len(seen)

for s_, exp in examples:
    print(edf_brute(s_), "(kỳ vọng", exp, ")")

5 (kỳ vọng 5 )
9 (kỳ vọng 9 )


**Độ phức tạp:** `O(n²)` xâu con × `O(n)` để đếm = `O(n³)` — `n = 1000` cho `~1.7·10⁸` thao tác Python: quá chậm. **Nút thắt:** đếm lại từ đầu mỗi xâu con. Hai xâu con cùng đầu `i` chỉ khác nhau **một ký tự cuối** ⇒ đếm tăng dần.

## 6. Ý tưởng thuật toán / bước đột phá

**Kiểm tra `O(1)`.** Với đầu `i` cố định, giữ `cnt[0..9]`, `k` = số chữ số có `cnt > 0`, `M = max cnt`. Thêm `d = s[j]`:
- **nếu** `cnt[d] == 0` **thì** `k += 1`;
- `cnt[d] += 1`; `M = max(M, cnt[d])`;
- xâu `s[i..j]` thỏa ⇔ `M · k == j − i + 1`.

**Khử trùng.**
- **Đơn giản:** thêm `s[i:j+1]` vào `set` — cắt chuỗi `O(L)` nhưng chạy trong C, đủ nhanh với `n = 1000`.
- **Không cắt chuỗi:** băm lăn `h = h·B + d` cùng vòng `j`, lưu `(h, L)` vào `set`.
- **Chắc chắn đúng, không băm:** trie hậu tố — đi theo `s[i], s[i+1], …` từ gốc; nút **mới tạo** hoặc **chưa từng được xét** tương ứng một xâu con mới.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng tiền tố, kiểm tra `O(10)`

In [7]:
def edf_prefix(s):
    n = len(s)
    pre = [[0] * (n + 1) for _ in range(10)]
    for i, ch in enumerate(s):
        d = ord(ch) - 48
        for x in range(10):
            pre[x][i + 1] = pre[x][i]
        pre[d][i + 1] += 1
    seen = set()
    for i in range(n):
        for j in range(i + 1, n + 1):
            vals = {pre[x][j] - pre[x][i] for x in range(10)} - {0}
            if len(vals) == 1:
                seen.add(s[i:j])
    return len(seen)

for s_, exp in examples:
    print(edf_prefix(s_), "(kỳ vọng", exp, ")")

5 (kỳ vọng 5 )
9 (kỳ vọng 9 )


**Độ phức tạp:** `O(n² · 10)` kiểm tra + chi phí cắt chuỗi. Hằng số 10 lớn trong Python — chậm hơn cách 3.

### 7.2 Cách 3 — Đếm tăng dần (`M · k == L`) + `set` xâu con

In [8]:
def edf_incremental(s):
    n = len(s)
    seen = set()
    for i in range(n):
        cnt = [0] * 10
        k = M = 0
        for j in range(i, n):
            d = ord(s[j]) - 48
            if cnt[d] == 0:
                k += 1
            cnt[d] += 1
            if cnt[d] > M:
                M = cnt[d]
            if M * k == j - i + 1:
                seen.add(s[i:j + 1])
    return len(seen)

for s_, exp in examples:
    print(edf_incremental(s_), "(kỳ vọng", exp, ")")

5 (kỳ vọng 5 )
9 (kỳ vọng 9 )


**Độ phức tạp:** `O(n²)` kiểm tra; cắt chuỗi chỉ cho xâu **thỏa** (thường là ít). Bộ nhớ `O(tổng độ dài các xâu thỏa khác nhau)`.

### 7.3 Cách 4 — Băm lăn `(hash, length)` thay cho cắt chuỗi

In [9]:
def edf_rolling(s, base=None, mod=(1 << 61) - 1, seed=2168):
    if base is None:
        base = random.Random(seed).randrange(10**6, mod - 1)
    n = len(s)
    seen = set()
    for i in range(n):
        cnt = [0] * 10
        k = M = 0
        h = 0
        for j in range(i, n):
            d = ord(s[j]) - 48
            h = (h * base + d + 1) % mod          # +1 để chữ số 0 không "biến mất"
            if cnt[d] == 0:
                k += 1
            cnt[d] += 1
            if cnt[d] > M:
                M = cnt[d]
            if M * k == j - i + 1:
                seen.add((h, j - i + 1))
    return len(seen)

for s_, exp in examples:
    print(edf_rolling(s_), "(kỳ vọng", exp, ")")

5 (kỳ vọng 5 )
9 (kỳ vọng 9 )


**Vì sao `+ 1`?** Nếu mã hóa chữ số `0` thành giá trị `0`, thì `"0"`, `"00"`, `"000"` đều có `h = 0` — chỉ khác nhau nhờ độ dài. Kèm độ dài đã đỡ, nhưng `"01"` và `"1"`… khác độ dài nên vẫn ổn; `+1` là thói quen an toàn cho mọi bảng chữ cái. **Độ phức tạp:** `O(n²)`, mỗi bước một phép nhân modulo số lớn.

### 7.4 Cách 5 — Trie các hậu tố (khử trùng chắc chắn, không băm)

In [10]:
def edf_trie(s):
    n = len(s)
    children = [[-1] * 10]           # trie dạng mảng: children[node][digit]
    counted = [False]
    total = 0
    for i in range(n):
        node = 0
        cnt = [0] * 10
        k = M = 0
        for j in range(i, n):
            d = ord(s[j]) - 48
            nxt = children[node][d]
            if nxt == -1:
                nxt = len(children)
                children[node][d] = nxt
                children.append([-1] * 10)
                counted.append(False)
            node = nxt
            if cnt[d] == 0:
                k += 1
            cnt[d] += 1
            if cnt[d] > M:
                M = cnt[d]
            if M * k == j - i + 1 and not counted[node]:
                counted[node] = True
                total += 1
    return total

for s_, exp in examples:
    print(edf_trie(s_), "(kỳ vọng", exp, ")")

5 (kỳ vọng 5 )
9 (kỳ vọng 9 )


**Độ phức tạp:** `O(n²)` thời gian; số nút trie = số xâu con khác nhau `≤ n(n+1)/2` (≈ 5·10⁵ với `n = 1000`) — mỗi nút một list 10 phần tử, tốn bộ nhớ đáng kể. **Ưu điểm:** không va chạm, không cắt chuỗi. Điều kiện thỏa **không phụ thuộc** vị trí xuất hiện (chỉ phụ thuộc nội dung xâu) nên đánh dấu ở nút là đúng.

## 8. TRACE từng bước

### 8.1 Trace cách 3 với `s = "1212"`

In [11]:
def trace_incremental(s):
    n = len(s); seen = set()
    print(f"  {'i':>2} {'j':>2} {'xâu':<6} {'k':>2} {'M':>2} {'L':>2}  thỏa?  mới?")
    for i in range(n):
        cnt = [0] * 10; k = M = 0
        for j in range(i, n):
            d = ord(s[j]) - 48
            if cnt[d] == 0: k += 1
            cnt[d] += 1; M = max(M, cnt[d])
            L = j - i + 1; ok = M * k == L
            t = s[i:j + 1]; new = ok and t not in seen
            if ok: seen.add(t)
            print(f"  {i:>2} {j:>2} {t:<6} {k:>2} {M:>2} {L:>2}  {str(ok):<5}  {'mới' if new else ('trùng' if ok else '')}")
    print("  => đáp án", len(seen), "| tập:", sorted(seen, key=lambda x: (len(x), x)))
    return len(seen)

_ = trace_incremental("1212")

   i  j xâu     k  M  L  thỏa?  mới?
   0  0 1       1  1  1  True   mới
   0  1 12      2  1  2  True   mới
   0  2 121     2  2  3  False  
   0  3 1212    2  2  4  True   mới
   1  1 2       1  1  1  True   mới
   1  2 21      2  1  2  True   mới
   1  3 212     2  2  3  False  
   2  2 1       1  1  1  True   trùng
   2  3 12      2  1  2  True   trùng
   3  3 2       1  1  1  True   trùng
  => đáp án 5 | tập: ['1', '2', '12', '21', '1212']


### 8.2 Trace trie với `s = "1010"` (ví dụ tự dựng — đã đối chiếu vét cạn, đối chiếu lại đề gốc)

In [12]:
def trace_trie(s):
    n = len(s); children = [[-1] * 10]; counted = [False]; total = 0
    for i in range(n):
        node = 0; cnt = [0] * 10; k = M = 0
        for j in range(i, n):
            d = ord(s[j]) - 48; nxt = children[node][d]; created = nxt == -1
            if created:
                nxt = len(children); children[node][d] = nxt; children.append([-1] * 10); counted.append(False)
            node = nxt
            if cnt[d] == 0: k += 1
            cnt[d] += 1; M = max(M, cnt[d])
            ok = M * k == j - i + 1
            note = ""
            if ok and not counted[node]:
                counted[node] = True; total += 1; note = f"ĐẾM (tổng {total})"
            elif ok:
                note = "thỏa nhưng nút đã đếm"
            print(f"  s[{i}:{j + 1}] = {s[i:j + 1]!r:<7} nút {node:>2} ({'tạo mới' if created else 'đã có '}) {note}")
    return total

r = trace_trie("1010")
print("=>", r, "| vét cạn:", edf_brute("1010"))

  s[0:1] = '1'     nút  1 (tạo mới) ĐẾM (tổng 1)
  s[0:2] = '10'    nút  2 (tạo mới) ĐẾM (tổng 2)
  s[0:3] = '101'   nút  3 (tạo mới) 
  s[0:4] = '1010'  nút  4 (tạo mới) ĐẾM (tổng 3)
  s[1:2] = '0'     nút  5 (tạo mới) ĐẾM (tổng 4)
  s[1:3] = '01'    nút  6 (tạo mới) ĐẾM (tổng 5)
  s[1:4] = '010'   nút  7 (tạo mới) 
  s[2:3] = '1'     nút  1 (đã có ) thỏa nhưng nút đã đếm
  s[2:4] = '10'    nút  2 (đã có ) thỏa nhưng nút đã đếm
  s[3:4] = '0'     nút  5 (đã có ) thỏa nhưng nút đã đếm
=> 5 | vét cạn: 5


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def equalDigitFrequency(self, s: str) -> int:
        n = len(s)
        seen = set()
        for i in range(n):
            cnt = [0] * 10
            distinct = 0                 # số chữ số khác nhau trong s[i..j]
            mx = 0                       # tần suất lớn nhất
            for j in range(i, n):
                d = ord(s[j]) - 48
                if cnt[d] == 0:
                    distinct += 1
                cnt[d] += 1
                if cnt[d] > mx:
                    mx = cnt[d]
                if mx * distinct == j - i + 1:       # mọi tần suất bằng mx
                    seen.add(s[i:j + 1])
        return len(seen)

for s_, exp in examples:
    assert Solution().equalDigitFrequency(s_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Điều kiện.** Gọi `c₁..c_k > 0` là tần suất các chữ số có mặt trong xâu dài `L`, `M = max cᵢ`. Ta có `L = Σcᵢ ≤ k·M`; dấu bằng ⇔ `M − cᵢ = 0` với mọi `i` ⇔ mọi tần suất bằng nhau. Bất biến của vòng `j`: `cnt`, `distinct`, `mx` đúng là các đại lượng của `s[i..j]` (thêm một ký tự chỉ có thể tăng `distinct` một đơn vị và tăng `mx` lên `cnt[d]`; `mx` không bao giờ giảm khi kéo dài xâu).

**Khử trùng.** Điều kiện chỉ phụ thuộc **nội dung** xâu con. `set` các xâu con (cách 3) đếm mỗi nội dung một lần. Trie (cách 5): mỗi nội dung ↔ đúng một nút, cờ `counted` đảm bảo đếm một lần. Băm lăn (cách 4): đúng **với xác suất rất cao** — sai chỉ khi hai xâu khác nhau cùng độ dài trùng băm modulo `2⁶¹ − 1`; với cơ số ngẫu nhiên, xác suất một cặp va chạm `≤ L/P` (định lý về số nghiệm đa thức), tổng trên `≤ 5·10⁵` xâu vẫn cực nhỏ.

**Đầy đủ.** Hai vòng lặp duyệt **mọi** cặp `(i, j)` ⇒ mọi xâu con đều được xét.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`, `D` = số xâu con thỏa khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. `Counter` mỗi xâu | `O(n³)` | `O(Σ độ dài xâu thỏa)` |
| 2. Tiền tố | `O(10 · n²)` + cắt chuỗi | `O(10 n)` + set |
| 3. Đếm tăng dần + set | `O(n²)` + cắt các xâu thỏa | set |
| 4. Băm lăn | `O(n²)` | `O(D)` cặp số |
| 5. Trie | `O(n²)` | `O(10 · số xâu con khác nhau)` |

In [14]:
rnd = random.Random(2168)
tests = {
    "ngẫu nhiên 10 chữ số": "".join(rnd.choice("0123456789") for _ in range(1000)),
    "chỉ '1','2'": "".join(rnd.choice("12") for _ in range(1000)),
    "toàn '7'": "7" * 1000,
}
for label_, s_ in tests.items():
    row = []
    for name, fn in [("incremental", edf_incremental), ("rolling", edf_rolling), ("Solution", Solution().equalDigitFrequency)]:
        t0 = time.perf_counter(); r = fn(s_); row.append(f"{name}={r} ({(time.perf_counter() - t0) * 1000:.0f} ms)")
    print(f"{label_:<22}: " + ", ".join(row))
s300 = tests["ngẫu nhiên 10 chữ số"][:300]
for name, fn in [("prefix", edf_prefix), ("trie", edf_trie), ("incremental", edf_incremental)]:
    t0 = time.perf_counter(); r = fn(s300); print(f"  n=300 {name:<11}: {r} ({(time.perf_counter() - t0) * 1000:.0f} ms)")

ngẫu nhiên 10 chữ số  : incremental=1638 (50 ms), rolling=1638 (107 ms), Solution=1638 (49 ms)


chỉ '1','2'           : incremental=16596 (62 ms), rolling=16596 (120 ms), Solution=16596 (60 ms)


toàn '7'              : incremental=1000 (132 ms), rolling=1000 (147 ms), Solution=1000 (175 ms)
  n=300 prefix     : 592 (37 ms)
  n=300 trie       : 592 (17 ms)
  n=300 incremental: 592 (4 ms)


## 12. Bộ test

In [15]:
ALL = {
    "prefix": edf_prefix,
    "incremental": edf_incremental,
    "rolling": edf_rolling,
    "trie": edf_trie,
    "Solution": lambda s: Solution().equalDigitFrequency(s),
}

def check(s, exp=None, brute=True, skip=()):
    ref = edf_brute(s) if brute else edf_incremental(s)
    for name, fn in ALL.items():
        if name in skip:
            continue
        got = fn(s)
        assert got == ref, (name, s, got, ref)
    if exp is not None:
        assert ref == exp, (s, ref, exp)
    return ref

for s_, exp in examples:
    check(s_, exp)
print("Ví dụ đề: OK")

check("7", 1)
check("77", 2)                       # "7", "77"
check("0000", 4)                     # mọi xâu con toàn '0' thỏa, 4 độ dài khác nhau
check("0123456789", 55)              # mọi xâu con có các chữ số khác nhau -> thỏa hết, 55 xâu khác nhau
check("1122", 6)                     # "1","2","11","22","12","1122"
check("1010", None)
print("Edge case (kỳ vọng tự dựng đã đối chiếu vét cạn): OK")

rnd = random.Random(2021)
for _ in range(600):
    n = rnd.randint(1, 14)
    alphabet = rnd.choice(["01", "012", "123", "0123456789"])
    check("".join(rnd.choice(alphabet) for _ in range(n)))
print("600 xâu ngẫu nhiên nhỏ so với vét cạn: OK")

for _ in range(4):
    n = rnd.randint(200, 1000)
    check("".join(rnd.choice("0123456789"[: rnd.randint(1, 10)]) for _ in range(n)), brute=False, skip=("prefix", "trie"))
print("4 xâu lớn (đối chiếu chéo incremental / rolling / Solution): OK")

Ví dụ đề: OK
Edge case (kỳ vọng tự dựng đã đối chiếu vét cạn): OK
600 xâu ngẫu nhiên nhỏ so với vét cạn: OK


4 xâu lớn (đối chiếu chéo incremental / rolling / Solution): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên khử trùng (đếm theo vị trí)

In [16]:
def bug_no_dedup(s):
    n = len(s); total = 0
    for i in range(n):
        cnt = [0] * 10; k = M = 0
        for j in range(i, n):
            d = ord(s[j]) - 48
            if cnt[d] == 0: k += 1
            cnt[d] += 1; M = max(M, cnt[d])
            if M * k == j - i + 1:
                total += 1                     # SAI: đếm cả các xâu trùng nội dung
    return total

print("'1212' -> sai:", bug_no_dedup("1212"), "| đúng:", Solution().equalDigitFrequency("1212"))

'1212' -> sai: 8 | đúng: 5


**Sửa:** `set` / băm / trie. Đề nói rõ "**unique** substrings".

### Lỗi 2 — So sánh tần suất của **cả 10** chữ số (kể cả chữ số không có mặt)

In [17]:
def bug_all_ten(s):
    seen = set(); n = len(s)
    for i in range(n):
        cnt = [0] * 10
        for j in range(i, n):
            cnt[ord(s[j]) - 48] += 1
            if len(set(cnt)) == 1:             # SAI: chữ số vắng mặt có cnt = 0
                seen.add(s[i:j + 1])
    return len(seen)

print("'1212' -> sai:", bug_all_ten("1212"), "| đúng:", Solution().equalDigitFrequency("1212"))

'1212' -> sai: 0 | đúng: 5


**Vì sao sai:** đòi hỏi cả 10 chữ số cùng tần suất — chỉ thỏa khi xâu chứa đủ 0–9 cùng số lần. **Sửa:** chỉ xét chữ số **có mặt** (`set(cnt) − {0}`) hoặc dùng `M · k == L`.

### Lỗi 3 — Không khởi tạo lại bộ đếm cho mỗi đầu `i`

In [18]:
def bug_no_reset(s):
    seen = set(); n = len(s)
    cnt = [0] * 10; k = M = 0                  # SAI: đặt ngoài vòng i
    for i in range(n):
        for j in range(i, n):
            d = ord(s[j]) - 48
            if cnt[d] == 0: k += 1
            cnt[d] += 1; M = max(M, cnt[d])
            if M * k == j - i + 1:
                seen.add(s[i:j + 1])
    return len(seen)

print("'12321' -> sai:", bug_no_reset("12321"), "| đúng:", Solution().equalDigitFrequency("12321"))

'12321' -> sai: 3 | đúng: 9


**Sửa:** `cnt`, `k`, `M` phải khởi tạo lại **trong** vòng `i`.

### Lỗi 4 — Kiểm `M · k == L` nhưng cập nhật `k` **sau** khi tăng `cnt`

In [19]:
def bug_distinct_order(s):
    seen = set(); n = len(s)
    for i in range(n):
        cnt = [0] * 10; k = M = 0
        for j in range(i, n):
            d = ord(s[j]) - 48
            cnt[d] += 1
            if cnt[d] == 0:                    # SAI: không bao giờ đúng sau khi đã tăng
                k += 1
            M = max(M, cnt[d])
            if M * k == j - i + 1:
                seen.add(s[i:j + 1])
    return len(seen)

print("'1212' -> sai:", bug_distinct_order("1212"), "| đúng:", Solution().equalDigitFrequency("1212"))

'1212' -> sai: 0 | đúng: 5


**Sửa:** kiểm `cnt[d] == 0` **trước** khi tăng (hoặc kiểm `cnt[d] == 1` sau khi tăng).

### Lỗi 5 — Băm với modulo nhỏ và không kèm độ dài ⇒ va chạm

In [20]:
def bug_small_hash(s, mod=101, base=10):
    seen = set(); n = len(s)
    for i in range(n):
        cnt = [0] * 10; k = M = 0; h = 0
        for j in range(i, n):
            d = ord(s[j]) - 48
            h = (h * base + d) % mod           # SAI: mod nhỏ, base = 10, không +1, không kèm độ dài
            if cnt[d] == 0: k += 1
            cnt[d] += 1; M = max(M, cnt[d])
            if M * k == j - i + 1:
                seen.add(h)
    return len(seen)

rnd_bad = random.Random(9)
s_bad = "".join(rnd_bad.choice("0123456789") for _ in range(120))
print("xâu ngẫu nhiên 120 ký tự -> băm nhỏ (sai):", bug_small_hash(s_bad), "| đúng:", Solution().equalDigitFrequency(s_bad))
print("ngay cả '0' và '00':", bug_small_hash("00"), "| đúng:", Solution().equalDigitFrequency("00"))

xâu ngẫu nhiên 120 ký tự -> băm nhỏ (sai): 96 | đúng: 302
ngay cả '0' và '00': 1 | đúng: 2


**Vì sao sai:** với `base = 10` và mã hóa chữ số trực tiếp, `h` thực chất là **giá trị số của xâu modulo 101** — mọi xâu có cùng giá trị số mod 101 bị gộp (ví dụ `"7"`, `"07"`, `"007"`, `"108"`…). Chỉ có 101 giá trị băm trong khi số xâu thỏa khác nhau lớn hơn nhiều ⇒ va chạm hàng loạt, kết quả bị đếm thiếu; `"0"` và `"00"` đều băm thành 0. **Sửa:** `P = 2⁶¹ − 1`, cơ số ngẫu nhiên lớn, mã hóa `d + 1`, lưu cặp `(h, L)` — hoặc dùng `set` xâu / trie khi `n` nhỏ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n ≤ 1000` + "xâu con" ⇒ `O(n²)` duyệt `(i, j)` với **cập nhật tăng dần** là chuẩn.
- "Mọi phần tử có cùng tần suất" ⇒ `max × số loại == độ dài`.
- "Xâu con **khác nhau**" ⇒ `set` xâu (n nhỏ), băm lăn (cần tiết kiệm), trie / suffix automaton (cần chính xác tuyệt đối, n lớn hơn).
- Bảng chữ cái nhỏ (10 chữ số) ⇒ mảng đếm 10 ô thay cho `dict`.
- Test có xâu toàn một ký tự (`"0000"`) và xâu toàn ký tự khác nhau (`"0123456789"`) — hai cực của bài.

## 15. Biến thể của bài

1. **Đếm theo vị trí** (không khử trùng) — chính là lỗi 1, cũng là một bài hợp lệ.
2. **LeetCode 1207 — Unique Number of Occurrences:** kiểm tần suất **khác nhau** (ngược lại).
3. **LeetCode 2083 — Substrings That Begin and End With the Same Letter**.
4. **LeetCode 1044 — Longest Duplicate Substring:** băm lăn + tìm nhị phân độ dài.
5. **Xâu con "cân bằng" độ dài tối đa** — xâu con dài nhất thỏa điều kiện (bài tập 5).

In [21]:
def count_distinct_substrings_trie(s):
    """Số xâu con khác nhau = số nút trie hậu tố (trừ gốc) — nền của cách 5."""
    root = {}; nodes = 0
    for i in range(len(s)):
        cur = root
        for ch in s[i:]:
            if ch not in cur:
                cur[ch] = {}; nodes += 1
            cur = cur[ch]
    return nodes

rnd_v = random.Random(15)
for _ in range(300):
    t = "".join(rnd_v.choice("012") for _ in range(rnd_v.randint(1, 12)))
    assert count_distinct_substrings_trie(t) == len({t[i:j] for i in range(len(t)) for j in range(i + 1, len(t) + 1)})
print("Số xâu con khác nhau của '1212':", count_distinct_substrings_trie("1212"), "| khớp vét cạn trên 300 xâu")

Số xâu con khác nhau của '1212': 7 | khớp vét cạn trên 300 xâu


## 16. So sánh các cách làm

| Cách | Kiểm tra | Khử trùng | Time | Khi nào dùng |
|---|---|---|---|---|
| 1. `Counter` | đếm lại | set xâu | `O(n³)` | kiểm thử |
| 2. Tiền tố | `O(10)` | set xâu | `O(10 n²)` | khi cần truy vấn đoạn tùy ý |
| 3. Tăng dần | `O(1)` | set xâu | `O(n²)` + cắt | **nộp bài** |
| 4. Băm lăn | `O(1)` | `(h, L)` | `O(n²)` | tiết kiệm bộ nhớ / ngôn ngữ khác |
| 5. Trie | `O(1)` | nút trie | `O(n²)` | chính xác tuyệt đối |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().equalDigitFrequency("112233")`?

**Bài 2 (tính tay):** Với `s = "1221"`, liệt kê mọi xâu con khác nhau và đánh dấu xâu thỏa.

**Bài 3 (điền TODO):** Hoàn thiện `edf_positions(s)` — đếm **theo vị trí** (không khử trùng), `O(n²)`.

**Bài 4 (tự viết lại):** Viết lại cách 4 dùng **băm kép** (hai modulo `10⁹+7` và `998244353`) thay vì `2⁶¹ − 1`.

**Bài 5 (mini challenge):** `longest_equal_freq(s)` — độ dài xâu con dài nhất thỏa điều kiện. Thử làm nhanh hơn `O(n²)` trong trường hợp chỉ có hai loại chữ số (gợi ý: hiệu số đếm + hashmap vị trí đầu tiên, như bài "Contiguous Array" 525 — cẩn thận: xâu chỉ chứa **một** loại chữ số cũng thỏa).

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def edf_positions(s):
    # TODO: như code nộp nhưng total += 1 thay vì seen.add
    pass

# Bài 4 — TODO
def edf_double_hash(s):
    # TODO: hai giá trị băm h1, h2 cùng cập nhật; lưu (h1, h2, L)
    pass

# Bài 5 — TODO
def longest_equal_freq(s):
    # TODO: O(n^2) tổng quát; trường hợp 2 chữ số: prefix diff + first occurrence
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Xâu một chữ số: `1, 2, 3, 11, 22, 33`.
- Mức 2: Hai loại: `12, 23` (mỗi chữ số 1 lần), `1122, 2233` (mỗi chữ số 2 lần); ba loại: chỉ `112233` (xâu `123` không xuất hiện liền nhau trong `s`).
- Mức 3: Tổng `6 + 4 + 1 = 11`.

**Bài 2**
- Mức 1: Xâu con khác nhau: `1, 2, 12, 22, 21, 122, 221, 1221`.
- Mức 2: `122`, `221` không thỏa.
- Mức 3: Còn 6.

**Bài 3**
- Mức 1: Bỏ `set`.
- Mức 2: `total += 1` khi thỏa.
- Mức 3: `"1212"` cho 8.

**Bài 4**
- Mức 1: `h1 = (h1·B1 + d + 1) % M1`, tương tự `h2`.
- Mức 2: Lưu bộ ba `(h1, h2, L)`.
- Mức 3: So với code nộp trên xâu ngẫu nhiên.

**Bài 5**
- Mức 1: Bản `O(n²)`: theo dõi `M · k == L`, cập nhật max độ dài.
- Mức 2: Hai chữ số `a, b`: đoạn thỏa nếu `#a == #b` (cả hai có mặt) **hoặc** đoạn chỉ gồm một chữ số.
- Mức 3: `#a − #b` tiền tố + vị trí đầu tiên; đoạn một chữ số = run dài nhất.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = Solution().equalDigitFrequency("112233")
print("Bài 1:", r1); assert r1 == edf_brute("112233")

# Bài 2
s2 = "1221"
subs = sorted({s2[i:j] for i in range(4) for j in range(i + 1, 5)}, key=lambda x: (len(x), x))
print("Bài 2:", [(t, len(set(Counter(t).values())) == 1) for t in subs])
assert Solution().equalDigitFrequency(s2) == 6

# Bài 3
def edf_positions_sol(s):
    n = len(s); total = 0
    for i in range(n):
        cnt = [0] * 10; k = M = 0
        for j in range(i, n):
            d = ord(s[j]) - 48
            if cnt[d] == 0: k += 1
            cnt[d] += 1; M = max(M, cnt[d])
            total += M * k == j - i + 1
    return total

rnd = random.Random(3)
for _ in range(400):
    t = "".join(rnd.choice("0123") for _ in range(rnd.randint(1, 12)))
    assert edf_positions_sol(t) == sum(len(set(Counter(t[i:j]).values())) == 1 for i in range(len(t)) for j in range(i + 1, len(t) + 1))
print("Bài 3 OK, '1212' ->", edf_positions_sol("1212"))

Bài 1: 11
Bài 2: [('1', True), ('2', True), ('12', True), ('21', True), ('22', True), ('122', False), ('221', False), ('1221', True)]
Bài 3 OK, '1212' -> 8


In [24]:
# Bài 4
def edf_double_hash_sol(s, B1=131, B2=137, M1=10**9 + 7, M2=998244353):
    n = len(s); seen = set()
    for i in range(n):
        cnt = [0] * 10; k = M = 0; h1 = h2 = 0
        for j in range(i, n):
            d = ord(s[j]) - 48
            h1 = (h1 * B1 + d + 1) % M1; h2 = (h2 * B2 + d + 1) % M2
            if cnt[d] == 0: k += 1
            cnt[d] += 1; M = max(M, cnt[d])
            if M * k == j - i + 1:
                seen.add((h1, h2, j - i + 1))
    return len(seen)

# Bài 5
def longest_equal_freq_sol(s):
    n = len(s); best = 0
    for i in range(n):
        cnt = [0] * 10; k = M = 0
        for j in range(i, n):
            d = ord(s[j]) - 48
            if cnt[d] == 0: k += 1
            cnt[d] += 1; M = max(M, cnt[d])
            if M * k == j - i + 1:
                best = max(best, j - i + 1)
    return best

def longest_two_digits(s):
    """Chỉ dùng khi s gồm đúng hai loại chữ số a, b: O(n)."""
    a, b = sorted(set(s))
    first = {0: -1}; diff = 0; best = 0
    for j, ch in enumerate(s):
        diff += 1 if ch == a else -1
        if diff in first:
            best = max(best, j - first[diff])
        else:
            first[diff] = j
    run = cur = 0                                  # đoạn chỉ một loại chữ số
    for j, ch in enumerate(s):
        cur = cur + 1 if j and s[j - 1] == ch else 1
        run = max(run, cur)
    return max(best, run)

rnd = random.Random(5)
for _ in range(400):
    t = "".join(rnd.choice("0123456789"[: rnd.randint(1, 10)]) for _ in range(rnd.randint(1, 60)))
    assert edf_double_hash_sol(t) == Solution().equalDigitFrequency(t)
    u = "".join(rnd.choice("47") for _ in range(rnd.randint(2, 40)))
    if len(set(u)) == 2:
        assert longest_two_digits(u) == longest_equal_freq_sol(u)
print("Bài 4, Bài 5 OK; ví dụ:", longest_equal_freq_sol("12321"), longest_two_digits("4477747"))

Bài 4, Bài 5 OK; ví dụ: 3 6


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "mọi xâu con, cập nhật tăng dần":**

```python
seen = set()
for i in range(n):
    cnt = [0] * 10; distinct = mx = 0
    for j in range(i, n):
        d = ord(s[j]) - 48
        if cnt[d] == 0: distinct += 1
        cnt[d] += 1; mx = max(mx, cnt[d])
        if mx * distinct == j - i + 1:
            seen.add(s[i:j + 1])
return len(seen)
```

**Checklist:**
- [ ] Khử trùng theo **nội dung** xâu?
- [ ] Chỉ xét chữ số **có mặt** (không so cả 10 ô)?
- [ ] Khởi tạo lại bộ đếm cho mỗi `i`?
- [ ] Cập nhật `distinct` **trước** khi tăng `cnt`?
- [ ] Nếu dùng băm: modulo lớn, kèm độ dài, mã hóa `d + 1`?
- [ ] Chữ ký đúng: `def equalDigitFrequency(self, s: str) -> int`? (đối chiếu đề gốc nếu có Premium)